## RAG System Architecture

### High-Level Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                         RAG SYSTEM                              │
└─────────────────────────────────────────────────────────────────┘

┌──────────────────┐
│   INDEXING PHASE │
└──────────────────┘

Documents (PDF/TXT)
        ↓
┌───────────────────┐
│ Document Chunker  │  ← Split into semantic chunks (200-500 tokens)
└───────────────────┘
        ↓
┌───────────────────┐
│ Embedding Model   │  ← Convert text to vectors (1536 dimensions)
└───────────────────┘
        ↓
┌───────────────────┐
│  Vector Database  │  ← Store in FAISS index
│      (FAISS)      │
└───────────────────┘

┌──────────────────┐
│  RETRIEVAL PHASE │
└──────────────────┘

User Query
        ↓
┌───────────────────┐
│ Query Embedding   │  ← Convert query to vector
└───────────────────┘
        ↓
┌───────────────────┐
│ Similarity Search │  ← Find top-k similar chunks (cosine/L2)
└───────────────────┘
        ↓
┌───────────────────┐
│ Context Retrieval │  ← Get top 3-5 relevant chunks
└───────────────────┘

┌──────────────────┐
│ GENERATION PHASE │
└──────────────────┘

Retrieved Context + User Query
        ↓
┌───────────────────┐
│ Prompt Template   │  ← Structure prompt with context
└───────────────────┘
        ↓
┌───────────────────┐
│  LLM (GPT/Llama)  │  ← Generate contextual answer
└───────────────────┘
        ↓
Final Answer + Sources
```

---

### Component Details

#### 1. **Document Processing Layer**
- **Input**: Raw documents (PDF, TXT, DOCX)
- **Chunking Strategy**:
  - Fixed-size chunks with overlap (recommended: 400 tokens, 50 token overlap)
  - Sentence-based chunking for better semantic boundaries
- **Output**: List of text chunks with metadata (source, page number, etc.)

#### 2. **Embedding & Vector Store Layer**
- **Embedding Model Options**:
  - OpenAI: `text-embedding-3-small` (1536 dim)
  - Hugging Face: `sentence-transformers/all-MiniLM-L6-v2` (384 dim)
  - Hugging Face: `BAAI/bge-small-en-v1.5` (384 dim)
- **Vector Database**:
  - FAISS (Facebook AI Similarity Search) - Free, fast, local
  - ChromaDB - Lightweight alternative
  - Pinecone - Managed cloud option (requires API key)

#### 3. **Retrieval Layer**
- **Similarity Metrics**:
  - Cosine Similarity (recommended for normalized vectors)
  - Euclidean Distance (L2)
  - Dot Product
- **Top-K Selection**: Typically retrieve 3-5 most relevant chunks
- **Optional Reranking**: Use cross-encoder for better precision

#### 4. **Generation Layer**
- **LLM Options**:
  - OpenAI: `gpt-4o-mini`, `gpt-3.5-turbo`
  - Hugging Face: `meta-llama/Llama-3.2-3B-Instruct`, `mistralai/Mistral-7B-Instruct-v0.2`
  - Google: `gemini-1.5-flash` (via API)
- **Prompt Engineering**:
  - System prompt defining role and constraints
  - Context injection
  - Few-shot examples (optional)

---

## Part 1: Environment Setup

### Tasks:
1. Install all required libraries
2. Import necessary modules
3. Configure API keys securely

### Libraries to Install:
- `gradio` - For building the UI
- `faiss-cpu` or `chromadb` - Vector database
- `sentence-transformers` - For embeddings (Hugging Face)
- `openai` - If using OpenAI API
- `langchain` - Optional, for easier RAG implementation
- `pypdf` or `PyPDF2` - For PDF processing
- `huggingface_hub` - For deployment

### Instructions:
```python
# TODO: Install required packages using pip
# Hint: Use !pip install -q <package_name>
```

### API Key Management:
- Use `getpass` for secure key input in Colab
- For Hugging Face deployment, use Secrets in Space settings
- Never hardcode API keys in your code!

---

In [1]:
# TODO: Install required packages

!pip install -q gradio
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q huggingface_hub
!pip install -q transformers
!pip install -q torch



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 10.9 MB/s eta 0:00:00


In [2]:
# TODO: Import necessary libraries
# Your code here

# Example imports you'll need:
# import gradio as gr
# import faiss
# from sentence_transformers import SentenceTransformer
# import numpy as np
# etc.

# Import necessary modules
import os
import gradio as gr
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from huggingface_hub import InferenceClient
from getpass import getpass

# Configure API key securely (Hugging Face Inference API - FREE)
HF_TOKEN = getpass("HF_TOKEN: ")
os.environ["HF_TOKEN"] = HF_TOKEN

print("✅ All libraries installed and imported successfully!")


HF_TOKEN: ··········
✅ All libraries installed and imported successfully!


## Part 2: Document Processing

### Objective:
Create a document processing pipeline that:
1. Loads documents from various sources (text files, PDFs)
2. Splits documents into manageable chunks
3. Preserves metadata (source, page numbers)

### Chunking Strategies:

#### Strategy 1: Fixed-Size Chunking
```
Document: "The quick brown fox jumps over the lazy dog. The dog was sleeping peacefully..."

Chunk Size: 10 words
Overlap: 2 words

Chunk 1: "The quick brown fox jumps over the lazy dog. The"
Chunk 2: "The dog was sleeping peacefully under the tree..."
           ↑↑ (overlap)
```

#### Strategy 2: Sentence-Based Chunking
```
Document: "Sentence 1. Sentence 2. Sentence 3. Sentence 4."

Sentences per chunk: 2

Chunk 1: "Sentence 1. Sentence 2."
Chunk 2: "Sentence 3. Sentence 4."
```

### Implementation Tips:
- **Recommended chunk size**: 200-500 tokens (roughly 150-400 words)
- **Overlap**: 10-20% of chunk size to maintain context
- **Metadata to track**:
  - Source file name
  - Page number (for PDFs)
  - Chunk index
  - Original position in document

### Tasks:
1. Create a `DocumentLoader` class to load text/PDF files
2. Create a `DocumentChunker` class with at least one chunking method
3. Test with sample documents

---

In [3]:
# TODO: Implement document loading functionality
# Your code here

# class DocumentLoader:
#     def load_txt(self, file_path):
#         pass
#
#     def load_pdf(self, file_path):
#         pass

class DocumentLoader:
    def load_txt(self, file_path):
        """Load a plain text file and return list of dicts with text + metadata"""
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        return [{"text": text, "source": file_path, "page": 1}]

    def load_pdf(self, file_path):
        """Load a PDF file page by page and return list of dicts with text + metadata"""
        reader = PdfReader(file_path)
        pages = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():  # skip empty pages
                pages.append({
                    "text": text,
                    "source": file_path,
                    "page": i + 1
                })
        return pages

    def load_from_text(self, raw_text, source_name="manual_input"):
        """Load directly from a raw string (useful for testing)"""
        return [{"text": raw_text, "source": source_name, "page": 1}]

print("✅ DocumentLoader defined!")

✅ DocumentLoader defined!


In [4]:
# TODO: Implement document chunking
# Your code here

# class DocumentChunker:
#     def chunk_by_tokens(self, text, chunk_size=400, overlap=50):
#         pass
#
#     def chunk_by_sentences(self, text, sentences_per_chunk=3):
#         pass

class DocumentChunker:
    def chunk_by_tokens(self, text, chunk_size=400, overlap=50):
        """Fixed-size word-based chunking with overlap"""
        words = text.split()
        chunks = []
        start = 0
        chunk_index = 0

        while start < len(words):
            end = start + chunk_size
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)
            chunks.append({
                "text": chunk_text,
                "chunk_index": chunk_index,
                "word_start": start,
                "word_end": end
            })
            start += chunk_size - overlap  # move forward with overlap
            chunk_index += 1

        return chunks

    def chunk_by_sentences(self, text, sentences_per_chunk=3):
        """Sentence-based chunking"""
        # Split on sentence endings
        import re
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        sentences = [s for s in sentences if s]  # remove empty

        chunks = []
        chunk_index = 0

        for i in range(0, len(sentences), sentences_per_chunk):
            chunk_sentences = sentences[i:i + sentences_per_chunk]
            chunk_text = " ".join(chunk_sentences)
            chunks.append({
                "text": chunk_text,
                "chunk_index": chunk_index,
                "sentence_start": i
            })
            chunk_index += 1

        return chunks

    def chunk_documents(self, documents, strategy="tokens", **kwargs):
        """Process a list of document dicts and return chunks with full metadata"""
        all_chunks = []
        for doc in documents:
            if strategy == "tokens":
                chunks = self.chunk_by_tokens(doc["text"], **kwargs)
            else:
                chunks = self.chunk_by_sentences(doc["text"], **kwargs)

            for chunk in chunks:
                chunk["source"] = doc["source"]
                chunk["page"] = doc["page"]
                all_chunks.append(chunk)

        return all_chunks

print("✅ DocumentChunker defined!")


✅ DocumentChunker defined!


## Part 3: Embeddings and Vector Database

### Understanding Embeddings

Embeddings convert text into numerical vectors that capture semantic meaning:

```
"The cat sat on the mat"  →  [0.234, -0.567, 0.123, ..., 0.891]  (1536 dimensions)
"A feline rested on the rug" →  [0.221, -0.543, 0.134, ..., 0.876]  (similar vector!)
"Python is a programming language" →  [-0.432, 0.765, -0.234, ..., 0.123]  (different!)
```

### Embedding Model Options:

#### Option 1: Hugging Face Models (FREE, Local)
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dimensions
embeddings = model.encode(["your text here"])
```

**Popular models**:
- `sentence-transformers/all-MiniLM-L6-v2` (384 dim) - Fast, lightweight
- `BAAI/bge-small-en-v1.5` (384 dim) - Better quality
- `sentence-transformers/all-mpnet-base-v2` (768 dim) - Highest quality

#### Option 2: OpenAI API (Paid, Cloud)
```python
from openai import OpenAI
client = OpenAI(api_key="your-key")
response = client.embeddings.create(
    input="your text",
    model="text-embedding-3-small"  # 1536 dimensions
)
```

### Vector Database: FAISS

FAISS (Facebook AI Similarity Search) is perfect for this assignment:
- Free and open-source
- Fast similarity search
- Easy to use
- Works locally (no API needed)

#### FAISS Index Types:

1. **IndexFlatL2** (Exact search, L2 distance)
   - Best for: Small datasets (<100k vectors)
   - Speed: Slower but exact
   
2. **IndexFlatIP** (Exact search, Inner Product/Cosine)
   - Best for: Normalized vectors, cosine similarity
   - Speed: Slower but exact

3. **IndexIVFFlat** (Approximate search)
   - Best for: Large datasets (>100k vectors)
   - Speed: Much faster, slightly less accurate

### Implementation Steps:

```python
# 1. Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Create FAISS index
dimension = 384  # Must match your embedding model
index = faiss.IndexFlatL2(dimension)

# 3. Generate embeddings for chunks
embeddings = embedding_model.encode(text_chunks)

# 4. Add to index
index.add(embeddings.astype('float32'))

# 5. Search
query_embedding = embedding_model.encode(["your query"])
distances, indices = index.search(query_embedding.astype('float32'), k=5)
```

### Tasks:
1. Choose and initialize an embedding model
2. Create a `VectorDatabase` class that:
   - Stores document chunks and their embeddings
   - Implements similarity search
   - Returns top-k relevant chunks
3. Test retrieval with sample queries

---

In [5]:
# TODO: Initialize embedding model
# Your code here

# Hint: Choose between Hugging Face (free) or OpenAI (paid)

# Initialize embedding model (FREE - runs locally, no API key needed)
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'  # 384 dimensions, fast & lightweight
EMBEDDING_DIMENSION = 384

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"✅ Embedding model '{EMBEDDING_MODEL_NAME}' loaded successfully!")
print(f"📐 Embedding dimension: {EMBEDDING_DIMENSION}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model 'all-MiniLM-L6-v2' loaded successfully!
📐 Embedding dimension: 384


In [6]:
# TODO: Implement Vector Database class
# Your code here

# class VectorDatabase:
#     def __init__(self, embedding_model, dimension):
#         pass
#
#     def add_documents(self, documents):
#         # Generate embeddings and add to FAISS index
#         pass
#
#     def search(self, query, k=5):
#         # Find top-k similar documents
#         pass
#
#     def save_index(self, path):
#         # Save FAISS index to disk
#         pass
#
#     def load_index(self, path):
#         # Load FAISS index from disk
#         pass

class VectorDatabase:
    def __init__(self, embedding_model, dimension=384):
        """Initialize FAISS index and storage"""
        self.embedding_model = embedding_model
        self.dimension = dimension
        self.index = faiss.IndexFlatL2(dimension)  # L2 distance, exact search
        self.documents = []  # store original chunk dicts

    def add_documents(self, documents):
        """Generate embeddings for chunks and add to FAISS index"""
        texts = [doc["text"] for doc in documents]

        print(f"⚙️ Generating embeddings for {len(texts)} chunks...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)
        embeddings = np.array(embeddings).astype('float32')

        self.index.add(embeddings)
        self.documents.extend(documents)

        print(f"✅ Added {len(texts)} chunks to vector database!")
        print(f"📦 Total vectors in index: {self.index.ntotal}")

    def search(self, query, k=5):
        """Find top-k most similar chunks to the query"""
        query_embedding = self.embedding_model.encode([query])
        query_embedding = np.array(query_embedding).astype('float32')

        # Search FAISS index
        distances, indices = self.index.search(query_embedding, k)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx != -1:  # -1 means not found
                result = self.documents[idx].copy()
                result["score"] = float(dist)
                results.append(result)

        return results

    def save_index(self, path="faiss_index"):
        """Save FAISS index and documents to disk"""
        faiss.write_index(self.index, f"{path}.index")
        np.save(f"{path}_docs.npy", np.array(self.documents, dtype=object))
        print(f"✅ Index saved to '{path}.index'")

    def load_index(self, path="faiss_index"):
        """Load FAISS index and documents from disk"""
        self.index = faiss.read_index(f"{path}.index")
        self.documents = list(np.load(f"{path}_docs.npy", allow_pickle=True))
        print(f"✅ Index loaded from '{path}.index'")
        print(f"📦 Total vectors in index: {self.index.ntotal}")

# Initialize the vector database
vector_db = VectorDatabase(embedding_model, dimension=EMBEDDING_DIMENSION)

print("✅ VectorDatabase initialized!")


✅ VectorDatabase initialized!


## Part 4: LLM Integration

### Choosing an LLM

You have several options for the generation component:

#### Option 1: OpenAI API (Recommended for Quality)
```python
from openai import OpenAI
client = OpenAI(api_key="your-key")

response = client.chat.completions.create(
    model="gpt-4o-mini",  # or gpt-3.5-turbo
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.7,
    max_tokens=500
)
```
**Pros**: High quality, reliable, fast  
**Cons**: Costs money (~$0.15 per 1M tokens for gpt-4o-mini)

#### Option 2: Hugging Face Inference API (FREE Tier Available)
```python
from huggingface_hub import InferenceClient
client = InferenceClient(token="your-hf-token")

response = client.text_generation(
    prompt,
    model="meta-llama/Llama-3.2-3B-Instruct",
    max_new_tokens=500
)
```
**Pros**: Free tier available, many models  
**Cons**: Rate limits, slower than paid APIs

#### Option 3: Google Gemini API (FREE Tier)
```python
import google.generativeai as genai
genai.configure(api_key="your-key")
model = genai.GenerativeModel('gemini-1.5-flash')
response = model.generate_content(prompt)
```
**Pros**: Free tier is generous, fast  
**Cons**: Different API structure

### Prompt Engineering for RAG

A good RAG prompt should:
1. **Define the role**: "You are a helpful assistant..."
2. **Provide context**: Include retrieved chunks
3. **Set constraints**: "Only use the provided context"
4. **Specify format**: How to structure the answer

#### Example Prompt Template:

```python
def create_rag_prompt(query, context_chunks):
    context = "\n\n".join([
        f"[Document {i+1}]\n{chunk}"
        for i, chunk in enumerate(context_chunks)
    ])
    
    prompt = f"""You are a helpful AI assistant. Answer the user's question based ONLY on the provided context.

IMPORTANT INSTRUCTIONS:
- Use only information from the context below
- If the context doesn't contain enough information, say "I don't have enough information in the provided context to answer this question."
- Be concise and direct
- Cite which document(s) you used: [Document 1], [Document 2], etc.

CONTEXT:
{context}

USER QUESTION: {query}

ANSWER:"""
    return prompt
```

### Response Quality Tips:

1. **Temperature**:
   - Low (0.1-0.3): More focused, deterministic
   - Medium (0.5-0.7): Balanced
   - High (0.8-1.0): More creative, varied

2. **Max Tokens**:
   - Set based on expected answer length
   - Typical range: 200-800 tokens

3. **System Message**:
   - Define behavior and tone
   - Set expertise level

### Tasks:
1. Choose and set up your LLM (OpenAI, Hugging Face, or Gemini)
2. Create a prompt template function
3. Implement a `generate_answer()` function
4. Test with sample queries and context

---

In [7]:
# TODO: Set up LLM client
# Your code here

# Choose one:
# - OpenAI API
# - Hugging Face Inference API
# - Google Gemini API

# Using Hugging Face Inference API (FREE tier)
from huggingface_hub import InferenceClient

# Initialize the client with your HF token
hf_client = InferenceClient(token=os.environ.get("HF_TOKEN"))

# We'll use Mistral 7B - free and good quality
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"✅ HuggingFace Inference client initialized!")
print(f"🤖 LLM Model: {LLM_MODEL}")


✅ HuggingFace Inference client initialized!
🤖 LLM Model: mistralai/Mistral-7B-Instruct-v0.3


In [8]:
# TODO: Implement prompt template and generation function
# Your code here

# def create_rag_prompt(query, context_chunks):
#     pass
#
# def generate_answer(query, context_chunks):
#     pass

def create_rag_prompt(query, context_chunks):
    """Build a RAG prompt from query and retrieved chunks"""
    context = "\n\n".join([
        f"[Document {i+1}] (Source: {chunk.get('source', 'unknown')}, Page: {chunk.get('page', 'N/A')})\n{chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    prompt = f"""<s>[INST] You are a helpful AI assistant. Answer the user's question based ONLY on the provided context.

IMPORTANT INSTRUCTIONS:
- Use only information from the context below
- If the context doesn't contain enough information, say "I don't have enough information in the provided context to answer this question."
- Be concise and direct
- Mention which document(s) you used, e.g. [Document 1]

CONTEXT:
{context}

USER QUESTION: {query}

ANSWER: [/INST]"""
    return prompt


def generate_answer(query, context_chunks, max_new_tokens=500, temperature=0.3):
    """Generate an answer using retrieved context chunks"""

    if not context_chunks:
        return "⚠️ No relevant context found. Please add documents to the knowledge base first."

    # Build the prompt
    prompt = create_rag_prompt(query, context_chunks)

    try:
        # Call HuggingFace Inference API
        response = hf_client.text_generation(
            prompt,
            model=LLM_MODEL,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            repetition_penalty=1.1,  # avoid repetitive text
            do_sample=True
        )
        return response.strip()

    except Exception as e:
        return f"❌ Error generating answer: {str(e)}\nTip: Check your HF token or try again in a moment."


print("✅ Prompt template and generate_answer() defined!")


✅ Prompt template and generate_answer() defined!


## Part 5: Complete RAG Pipeline

### Pipeline Architecture

Now you'll combine all components into a unified RAG system:

```
User Query
    ↓
[1] Query Validation & Preprocessing
    - Check query length
    - Remove special characters if needed
    - Apply safety filters
    ↓
[2] Retrieval
    - Convert query to embedding
    - Search vector database
    - Get top-k relevant chunks (k=3-5)
    ↓
[3] Context Preparation
    - Format retrieved chunks
    - Add metadata (source, relevance score)
    - Truncate if too long
    ↓
[4] Generation
    - Create prompt with context
    - Call LLM
    - Parse response
    ↓
[5] Post-processing
    - Format answer
    - Add source citations
    - Return to user
```

### RAG System Class Structure

```python
class RAGSystem:
    def __init__(self, vector_db, llm_client, embedding_model):
        self.vector_db = vector_db
        self.llm_client = llm_client
        self.embedding_model = embedding_model
    
    def validate_query(self, query):
        """Validate and preprocess user query"""
        pass
    
    def retrieve(self, query, k=5):
        """Retrieve relevant documents"""
        pass
    
    def generate(self, query, context):
        """Generate answer using LLM"""
        pass
    
    def query(self, user_query, k=5):
        """Main RAG pipeline"""
        # 1. Validate
        # 2. Retrieve
        # 3. Generate
        # 4. Format response
        pass
```

### Error Handling

Your system should gracefully handle:
- Empty queries
- No relevant documents found
- LLM API failures
- Rate limit errors
- Invalid input formats

```python
try:
    result = rag_system.query(user_query)
except ValueError as e:
    return {"error": "Invalid query", "message": str(e)}
except Exception as e:
    return {"error": "System error", "message": "Please try again"}
```

### Response Format

Structure your response as a dictionary:
```python
{
    "answer": "The generated answer text...",
    "sources": [
        {
            "text": "Retrieved chunk 1...",
            "metadata": {"source": "doc1.pdf", "page": 5},
            "score": 0.87
        },
        # ... more sources
    ],
    "num_sources": 3,
    "query_time": 1.23  # seconds
}
```

### Tasks:
1. Implement the `RAGSystem` class
2. Add proper error handling
3. Test the complete pipeline with various queries
4. Measure and optimize query response time

---

In [9]:
# TODO: Implement complete RAG System
# Your code here

# class RAGSystem:
#     def __init__(self, vector_db, llm_client, embedding_model):
#         pass
#
#     def query(self, user_query, k=5):
#         pass

# Using Hugging Face Inference API (FREE tier)
from huggingface_hub import InferenceClient

# Initialize the client with your HF token
hf_client = InferenceClient(token=os.environ.get("HF_TOKEN"))

# We'll use Mistral 7B - free and good quality
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"✅ HuggingFace Inference client initialized!")
print(f"🤖 LLM Model: {LLM_MODEL}")


✅ HuggingFace Inference client initialized!
🤖 LLM Model: mistralai/Mistral-7B-Instruct-v0.3


In [10]:
# TODO: Test RAG system with sample queries
# Your code here

# test_queries = [
#     "What is machine learning?",
#     "Explain neural networks",
#     "How does RAG work?"
# ]
#
# for query in test_queries:
#     result = rag_system.query(query)
#     print(f"Q: {query}")
#     print(f"A: {result['answer']}\n")

def create_rag_prompt(query, context_chunks):
    """Build a RAG prompt from query and retrieved chunks"""
    context = "\n\n".join([
        f"[Document {i+1}] (Source: {chunk.get('source', 'unknown')}, Page: {chunk.get('page', 'N/A')})\n{chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    prompt = f"""<s>[INST] You are a helpful AI assistant. Answer the user's question based ONLY on the provided context.

IMPORTANT INSTRUCTIONS:
- Use only information from the context below
- If the context doesn't contain enough information, say "I don't have enough information in the provided context to answer this question."
- Be concise and direct
- Mention which document(s) you used, e.g. [Document 1]

CONTEXT:
{context}

USER QUESTION: {query}

ANSWER: [/INST]"""
    return prompt


def generate_answer(query, context_chunks, max_new_tokens=500, temperature=0.3):
    """Generate an answer using retrieved context chunks"""

    if not context_chunks:
        return "⚠️ No relevant context found. Please add documents to the knowledge base first."

    # Build the prompt
    prompt = create_rag_prompt(query, context_chunks)

    try:
        # Call HuggingFace Inference API
        response = hf_client.text_generation(
            prompt,
            model=LLM_MODEL,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            repetition_penalty=1.1,  # avoid repetitive text
            do_sample=True
        )
        return response.strip()

    except Exception as e:
        return f"❌ Error generating answer: {str(e)}\nTip: Check your HF token or try again in a moment."


print("✅ Prompt template and generate_answer() defined!")


✅ Prompt template and generate_answer() defined!


## Part 6: Gradio User Interface

### Why Gradio?

Gradio is perfect for ML demos because it:
- Creates UIs with just a few lines of code
- Supports various input/output types
- Has built-in sharing capabilities
- Integrates seamlessly with Hugging Face Spaces
- Provides nice default styling

### Gradio Interface Components

#### Basic Structure:
```python
import gradio as gr

def chat(message, history):
    # Your RAG logic here
    response = rag_system.query(message)
    return response['answer']

# Create interface
demo = gr.ChatInterface(
    fn=chat,
    title="My RAG System",
    description="Ask questions about the uploaded documents"
)

demo.launch()
```

### Enhanced UI Features

#### 1. Document Upload
```python
with gr.Blocks() as demo:
    gr.Markdown("# RAG System")
    
    with gr.Tab("Upload Documents"):
        file_input = gr.File(
            label="Upload PDF or TXT",
            file_types=[".pdf", ".txt"]
        )
        upload_btn = gr.Button("Process Document")
        status = gr.Textbox(label="Status")
        
        upload_btn.click(
            fn=process_document,
            inputs=[file_input],
            outputs=[status]
        )
```

#### 2. Chat Interface with Sources
```python
    with gr.Tab("Chat"):
        chatbot = gr.Chatbot(height=400)
        msg = gr.Textbox(
            label="Your Question",
            placeholder="Type your question here..."
        )
        sources = gr.Accordion("📚 Sources", open=False)
        with sources:
            source_display = gr.Markdown()
```

#### 3. Configuration Panel
```python
    with gr.Tab("Settings"):
        num_sources = gr.Slider(
            minimum=1, maximum=10, value=5,
            label="Number of sources to retrieve"
        )
        temperature = gr.Slider(
            minimum=0, maximum=1, value=0.7,
            label="LLM Temperature"
        )
```

### UI Design Best Practices

1. **Clear Instructions**:
   - Add description of what the system does
   - Provide example queries
   - Explain limitations

2. **Visual Feedback**:
   - Show loading indicators
   - Display progress for document processing
   - Highlight sources used

3. **Error Messages**:
   - User-friendly error messages
   - Suggestions for fixing issues

4. **Source Attribution**:
   - Show which documents were used
   - Display relevance scores
   - Allow expanding to see full context

### Example Complete Interface:

```python
def respond(message, chat_history, num_sources):
    # Get RAG response
    result = rag_system.query(message, k=num_sources)
    
    # Format response
    answer = result['answer']
    
    # Format sources
    sources_text = "### Retrieved Sources:\n\n"
    for i, source in enumerate(result['sources'], 1):
        sources_text += f"**[{i}]** {source['text'][:200]}...\n\n"
    
    # Update chat history
    chat_history.append((message, answer))
    
    return "", chat_history, sources_text

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 RAG-Powered Q&A System")
    gr.Markdown("Ask questions about your uploaded documents!")
    
    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Your Question")
    with gr.Row():
        num_sources = gr.Slider(1, 10, 5, label="Sources")
        clear = gr.Button("Clear")
    
    sources_display = gr.Markdown()
    
    msg.submit(respond, [msg, chatbot, num_sources],
               [msg, chatbot, sources_display])
    clear.click(lambda: None, None, chatbot, queue=False)
```

### Tasks:
1. Create a Gradio interface for your RAG system
2. Add document upload functionality (optional but recommended)
3. Display sources alongside answers
4. Add configuration options (number of sources, temperature)
5. Test the interface locally in Colab

---

In [11]:
# TODO: Create Gradio interface
# Your code here

# import gradio as gr
#
# def chat_function(message, history):
#     # Call your RAG system
#     pass
#
# demo = gr.ChatInterface(
#     fn=chat_function,
#     title="RAG System",
#     examples=[
#         "What is machine learning?",
#         "Explain neural networks"
#     ]
# )
#
# demo.launch()

import gradio as gr

# ── Helper: process uploaded file and add to vector DB ──────────────────────
def process_document(file):
    """Load, chunk, and index an uploaded file"""
    if file is None:
        return "⚠️ No file uploaded."
    try:
        loader = DocumentLoader()
        ext = file.name.lower()

        if ext.endswith(".pdf"):
            docs = loader.load_pdf(file.name)
        elif ext.endswith(".txt"):
            docs = loader.load_txt(file.name)
        else:
            return "❌ Unsupported file type. Please upload a PDF or TXT file."

        chunker = DocumentChunker()
        chunks = chunker.chunk_documents(docs, strategy="tokens",
                                         chunk_size=400, overlap=50)
        vector_db.add_documents(chunks)

        return (f"✅ Processed **{file.name}**\n"
                f"- Pages/Sections: {len(docs)}\n"
                f"- Chunks created: {len(chunks)}\n"
                f"- Total vectors in DB: {vector_db.index.ntotal}")
    except Exception as e:
        return f"❌ Error processing document: {str(e)}"


# ── Helper: run full RAG pipeline ────────────────────────────────────────────
def rag_query(query, k=5, temperature=0.3):
    """Retrieve relevant chunks and generate answer"""
    if vector_db.index.ntotal == 0:
        return "⚠️ No documents indexed yet. Please upload a document first.", ""

    # Retrieve
    results = vector_db.search(query, k=k)

    # Generate
    answer = generate_answer(query, results,
                             max_new_tokens=500,
                             temperature=temperature)

    # Format sources
    sources_md = "### 📚 Retrieved Sources:\n\n"
    for i, r in enumerate(results, 1):
        sources_md += (f"**[{i}]** 📄 `{r.get('source','unknown')}` "
                       f"| Page {r.get('page','N/A')} "
                       f"| Score: {r.get('score', 0):.4f}\n\n"
                       f"> {r['text'][:250]}...\n\n---\n\n")

    return answer, sources_md


# ── Chat handler ─────────────────────────────────────────────────────────────
def respond(message, chat_history, num_sources, temperature):
    if not message.strip():
        return "", chat_history, ""

    answer, sources_md = rag_query(message,
                                   k=int(num_sources),
                                   temperature=temperature)
    chat_history.append((message, answer))
    return "", chat_history, sources_md


# ── Build Gradio UI ──────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="RAG Q&A System") as demo:

    gr.Markdown("# 🤖 RAG-Powered Q&A System")
    gr.Markdown("Upload your documents, then ask questions about them!")

    with gr.Tab("📂 Upload Documents"):
        file_input = gr.File(
            label="Upload PDF or TXT file",
            file_types=[".pdf", ".txt"]
        )
        upload_btn = gr.Button("⚙️ Process Document", variant="primary")
        upload_status = gr.Markdown(label="Status")

        upload_btn.click(
            fn=process_document,
            inputs=[file_input],
            outputs=[upload_status]
        )

    with gr.Tab("💬 Chat"):
        chatbot = gr.Chatbot(height=420, label="Conversation")

        with gr.Row():
            msg_box = gr.Textbox(
                label="Your Question",
                placeholder="e.g. What is the main topic of the document?",
                scale=4
            )
            send_btn = gr.Button("Send", variant="primary", scale=1)

        with gr.Row():
            num_sources = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label="Number of sources to retrieve"
            )
            temperature = gr.Slider(
                minimum=0.1, maximum=1.0, value=0.3, step=0.1,
                label="LLM Temperature"
            )

        clear_btn = gr.Button("🗑️ Clear Chat")
        sources_display = gr.Markdown(label="Sources")

        # Example queries
        gr.Examples(
            examples=[
                "What is the main topic of the document?",
                "Summarize the key points.",
                "What conclusions are drawn?",
                "Who are the main people or entities mentioned?",
            ],
            inputs=msg_box
        )

        # Wire up events
        msg_box.submit(
            fn=respond,
            inputs=[msg_box, chatbot, num_sources, temperature],
            outputs=[msg_box, chatbot, sources_display]
        )
        send_btn.click(
            fn=respond,
            inputs=[msg_box, chatbot, num_sources, temperature],
            outputs=[msg_box, chatbot, sources_display]
        )
        clear_btn.click(
            fn=lambda: ([], ""),
            outputs=[chatbot, sources_display]
        )

    with gr.Tab("ℹ️ About"):
        gr.Markdown("""
        ## How to use this RAG System

        1. **Upload** a PDF or TXT document in the *Upload Documents* tab
        2. Wait for processing confirmation
        3. Go to the **Chat** tab and ask questions
        4. Adjust **sources** slider to control how many chunks are retrieved
        5. Adjust **temperature** for more focused (low) or creative (high) answers

        ## Technologies Used
        - 🔍 **Embeddings**: `all-MiniLM-L6-v2` (Sentence Transformers)
        - 🗄️ **Vector DB**: FAISS (Facebook AI Similarity Search)
        - 🤖 **LLM**: Mistral-7B via HuggingFace Inference API
        - 🖥️ **UI**: Gradio
        """)

# Launch
demo.launch(debug=True, share=True)


/tmp/ipykernel_4767/2516486741.py:90: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="RAG Q&A System") as demo:
/tmp/ipykernel_4767/2516486741.py:110: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=420, label="Conversation")
/tmp/ipykernel_4767/2516486741.py:110: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=420, label="Conversation")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7c4c3e4d755b3057ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7c4c3e4d755b3057ba.gradio.live


## Part 7: Deployment to Hugging Face Spaces

### What is Hugging Face Spaces?

Hugging Face Spaces is a **free hosting platform** for ML demos that:
- Supports Gradio, Streamlit, and static HTML
- Provides free CPU and GPU resources
- Auto-deploys from Git repositories
- Offers easy sharing with unique URLs

### Deployment Steps

#### Step 1: Prepare Your Files

Create these files in your project:

**1. `app.py`** (Main application file)
```python
import gradio as gr
from sentence_transformers import SentenceTransformer
import faiss
# ... other imports

# Your RAG system code here

# Launch Gradio interface
if __name__ == "__main__":
    demo.launch()
```

**2. `requirements.txt`** (Dependencies)
```
gradio>=4.0.0
sentence-transformers>=2.2.0
faiss-cpu>=1.7.0
numpy>=1.24.0
# Add your other dependencies
```

**3. `README.md`** (Documentation)
```markdown
---
title: My RAG System
emoji: 🤖
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 4.0.0
app_file: app.py
pinned: false
---

# RAG System

A Retrieval-Augmented Generation system for Q&A.

## Features
- Semantic search over documents
- Contextual answer generation
- Source attribution
```

**4. `.gitignore`** (Optional)
```
__pycache__/
*.pyc
.env
.venv/
*.log
```

#### Step 2: Create a Space

1. Go to https://huggingface.co/spaces
2. Click **"Create new Space"**
3. Fill in:
   - Space name: `my-rag-system`
   - License: `mit` or `apache-2.0`
   - SDK: **Gradio**
   - Visibility: Public or Private
4. Click **"Create Space"**

#### Step 3: Upload Files

**Option A: Web Interface (Easiest)**
1. Click **"Files"** tab
2. Click **"Add file"** → **"Upload files"**
3. Upload `app.py`, `requirements.txt`, `README.md`
4. Click **"Commit changes"**

**Option B: Git (Recommended for developers)**
```bash
# Clone your Space
git clone https://huggingface.co/spaces/YOUR-USERNAME/my-rag-system
cd my-rag-system

# Add your files
cp /path/to/app.py .
cp /path/to/requirements.txt .

# Commit and push
git add .
git commit -m "Initial commit"
git push
```

**Option C: From Colab**
```python
from huggingface_hub import HfApi, create_repo

# Login (you'll need a Hugging Face token)
api = HfApi()
api.create_repo(
    repo_id="my-rag-system",
    repo_type="space",
    space_sdk="gradio"
)

# Upload files
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id="YOUR-USERNAME/my-rag-system",
    repo_type="space"
)
```

#### Step 4: Configure Secrets (for API Keys)

If you're using OpenAI or other APIs:

1. Go to Space **Settings** → **Variables and secrets**
2. Click **"New secret"**
3. Add:
   - Name: `OPENAI_API_KEY`
   - Value: Your API key
4. Click **"Save"**

In your `app.py`, access secrets:
```python
import os
api_key = os.environ.get("OPENAI_API_KEY")
```

#### Step 5: Monitor Deployment

1. Go to **"Logs"** tab to see build progress
2. Wait for **"Running on public URL"** message
3. Visit your Space URL: `https://huggingface.co/spaces/YOUR-USERNAME/my-rag-system`

### Deployment Checklist

✅ All dependencies in `requirements.txt`  
✅ No hardcoded API keys (use Secrets)  
✅ Proper error handling in `app.py`  
✅ Clear README with usage instructions  
✅ Example queries in Gradio interface  
✅ Appropriate model sizes (avoid huge models on free tier)  
✅ Graceful degradation if APIs fail  

### Common Issues & Solutions

**Issue**: "ModuleNotFoundError"
- **Solution**: Add missing package to `requirements.txt`

**Issue**: "Out of memory"
- **Solution**: Use smaller models or request GPU Space

**Issue**: "API rate limit exceeded"
- **Solution**: Add rate limiting or caching in your code

**Issue**: Space keeps restarting
- **Solution**: Check logs for errors, fix bugs in `app.py`

### Upgrading to GPU Space

For better performance:
1. Go to Space **Settings**
2. Under **"Hardware"**, select GPU (may require Hugging Face Pro)
3. Modify code to use GPU:
```python
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('model-name').to(device)
```

### Tasks:
1. Create a Hugging Face account (if you don't have one)
2. Package your code into `app.py`
3. Create `requirements.txt` with all dependencies
4. Deploy to Hugging Face Spaces
5. Test your deployed Space
6. Share the URL in your submission

---

In [12]:
# TODO: Prepare files for deployment
# Your code here

# Hint: Export your RAG system to app.py
# Create requirements.txt
# Write README.md
# ── 1. Write app.py ──────────────────────────────────────────────────────────
app_py_content = '''
import os
import numpy as np
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
from pypdf import PdfReader
import re

# ── Config ────────────────────────────────────────────────────────────────────
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIMENSION  = 384
LLM_MODEL            = "mistralai/Mistral-7B-Instruct-v0.3"
HF_TOKEN             = os.environ.get("HF_TOKEN", "")

# ── Init models ───────────────────────────────────────────────────────────────
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
hf_client       = InferenceClient(token=HF_TOKEN)

# ── Document classes ──────────────────────────────────────────────────────────
class DocumentLoader:
    def load_txt(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        return [{"text": text, "source": file_path, "page": 1}]

    def load_pdf(self, file_path):
        reader = PdfReader(file_path)
        pages  = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append({"text": text, "source": file_path, "page": i + 1})
        return pages

class DocumentChunker:
    def chunk_by_tokens(self, text, chunk_size=400, overlap=50):
        words   = text.split()
        chunks  = []
        start   = 0
        idx     = 0
        while start < len(words):
            chunk_text = " ".join(words[start:start + chunk_size])
            chunks.append({"text": chunk_text, "chunk_index": idx,
                           "word_start": start})
            start += chunk_size - overlap
            idx   += 1
        return chunks

    def chunk_documents(self, documents, strategy="tokens", **kwargs):
        all_chunks = []
        for doc in documents:
            chunks = self.chunk_by_tokens(doc["text"], **kwargs)
            for chunk in chunks:
                chunk["source"] = doc["source"]
                chunk["page"]   = doc["page"]
                all_chunks.append(chunk)
        return all_chunks

class VectorDatabase:
    def __init__(self, embedding_model, dimension=384):
        self.embedding_model = embedding_model
        self.dimension       = dimension
        self.index           = faiss.IndexFlatL2(dimension)
        self.documents       = []

    def add_documents(self, documents):
        texts      = [d["text"] for d in documents]
        embeddings = self.embedding_model.encode(texts)
        embeddings = np.array(embeddings).astype("float32")
        self.index.add(embeddings)
        self.documents.extend(documents)

    def search(self, query, k=5):
        qe = np.array(self.embedding_model.encode([query])).astype("float32")
        distances, indices = self.index.search(qe, k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx != -1:
                r          = self.documents[idx].copy()
                r["score"] = float(dist)
                results.append(r)
        return results

vector_db = VectorDatabase(embedding_model, EMBEDDING_DIMENSION)

# ── LLM helpers ───────────────────────────────────────────────────────────────
def create_rag_prompt(query, context_chunks):
    context = "\\n\\n".join([
        f"[Document {i+1}] (Source: {c.get(\'source\',\'unknown\')}, Page: {c.get(\'page\',\'N/A\')})"
        f"\\n{c[\'text\']}"
        for i, c in enumerate(context_chunks)
    ])
    return (f"<s>[INST] You are a helpful AI assistant. Answer ONLY from the context below.\\n\\n"
            f"If the context lacks enough info, say so clearly.\\n\\n"
            f"CONTEXT:\\n{context}\\n\\nQUESTION: {query}\\n\\nANSWER: [/INST]")

def generate_answer(query, context_chunks, max_new_tokens=500, temperature=0.3):
    if not context_chunks:
        return "No relevant context found. Please upload a document first."
    prompt = create_rag_prompt(query, context_chunks)
    try:
        return hf_client.text_generation(
            prompt, model=LLM_MODEL,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            repetition_penalty=1.1,
            do_sample=True
        ).strip()
    except Exception as e:
        return f"Error generating answer: {e}"

# ── Gradio helpers ────────────────────────────────────────────────────────────
def process_document(file):
    if file is None:
        return "No file uploaded."
    try:
        loader  = DocumentLoader()
        chunker = DocumentChunker()
        docs    = loader.load_pdf(file.name) if file.name.endswith(".pdf") else loader.load_txt(file.name)
        chunks  = chunker.chunk_documents(docs, strategy="tokens", chunk_size=400, overlap=50)
        vector_db.add_documents(chunks)
        return (f"Processed {file.name} — "
                f"{len(docs)} page(s), {len(chunks)} chunks, "
                f"{vector_db.index.ntotal} total vectors.")
    except Exception as e:
        return f"Error: {e}"

def respond(message, chat_history, num_sources, temperature):
    if not message.strip():
        return "", chat_history, ""
    results    = vector_db.search(message, k=int(num_sources))
    answer     = generate_answer(message, results, temperature=temperature)
    sources_md = "### Sources\\n\\n"
    for i, r in enumerate(results, 1):
        sources_md += (f"**[{i}]** `{r.get(\'source\',\'unknown\')}` "
                       f"Page {r.get(\'page\',\'N/A\')} Score {r.get(\'score\',0):.4f}\\n\\n"
                       f"> {r[\'text\'][:250]}...\\n\\n---\\n\\n")
    chat_history.append((message, answer))
    return "", chat_history, sources_md

# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="RAG Q&A System") as demo:
    gr.Markdown("# RAG-Powered Q&A System")
    gr.Markdown("Upload a document, then ask questions about it.")

    with gr.Tab("Upload Documents"):
        file_input    = gr.File(label="Upload PDF or TXT", file_types=[".pdf", ".txt"])
        upload_btn    = gr.Button("Process Document", variant="primary")
        upload_status = gr.Markdown()
        upload_btn.click(process_document, inputs=[file_input], outputs=[upload_status])

    with gr.Tab("Chat"):
        chatbot  = gr.Chatbot(height=420)
        msg_box  = gr.Textbox(label="Your Question",
                              placeholder="Ask something about your document...")
        send_btn = gr.Button("Send", variant="primary")
        with gr.Row():
            num_sources = gr.Slider(1, 10, 5, step=1, label="Sources to retrieve")
            temperature = gr.Slider(0.1, 1.0, 0.3, step=0.1, label="Temperature")
        clear_btn       = gr.Button("Clear Chat")
        sources_display = gr.Markdown()
        gr.Examples(["Summarize the document.", "What are the key points?",
                     "Who is mentioned in the document?", "What conclusions are drawn?"],
                    inputs=msg_box)
        msg_box.submit(respond, [msg_box, chatbot, num_sources, temperature],
                       [msg_box, chatbot, sources_display])
        send_btn.click(respond, [msg_box, chatbot, num_sources, temperature],
                       [msg_box, chatbot, sources_display])
        clear_btn.click(lambda: ([], ""), outputs=[chatbot, sources_display])

if __name__ == "__main__":
    demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_py_content)
print("✅ app.py created!")


# ── 2. Write requirements.txt ─────────────────────────────────────────────────
requirements = """gradio>=4.0.0
sentence-transformers>=2.2.0
faiss-cpu>=1.7.0
numpy>=1.24.0
pypdf>=3.0.0
huggingface_hub>=0.20.0
transformers>=4.35.0
torch>=2.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("✅ requirements.txt created!")


# ── 3. Write README.md ────────────────────────────────────────────────────────
readme = """---
title: RAG Q&A System
emoji: 🤖
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 4.0.0
app_file: app.py
pinned: false
---

# RAG-Powered Q&A System

A Retrieval-Augmented Generation (RAG) system that answers questions about uploaded documents.

## How to Use
1. Go to the **Upload Documents** tab and upload a PDF or TXT file
2. Wait for the processing confirmation
3. Go to the **Chat** tab and ask questions about your document
4. Adjust the sliders to control retrieval and creativity

## Technologies Used
- **Embeddings**: `all-MiniLM-L6-v2` (Sentence Transformers, runs locally)
- **Vector DB**: FAISS (Facebook AI Similarity Search)
- **LLM**: Mistral-7B-Instruct via Hugging Face Inference API
- **UI**: Gradio

## Example Queries
- Summarize the document.
- What are the key points?
- Who is mentioned in the document?
- What conclusions are drawn?

## Setup (for HF Spaces)
Add your `HF_TOKEN` in Space Settings → Variables and Secrets.
"""

with open("README.md", "w") as f:
    f.write(readme)
print("✅ README.md created!")


# ── 4. Write .gitignore ───────────────────────────────────────────────────────
gitignore = """__pycache__/
*.pyc
.env
*.log
faiss_index*
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)
print("✅ .gitignore created!")

print("\n🎉 All deployment files ready!")

✅ app.py created!
✅ requirements.txt created!
✅ README.md created!
✅ .gitignore created!

🎉 All deployment files ready!


## Part 8: Testing and Evaluation

### Why Evaluate Your RAG System?

Testing ensures your system:
- Retrieves relevant documents
- Generates accurate answers
- Handles edge cases properly
- Performs efficiently

### Evaluation Metrics

#### 1. Retrieval Quality

**Precision@K**: What percentage of top-K retrieved docs are relevant?
```python
def precision_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = retrieved_docs[:k]
    relevant_count = sum(1 for doc in retrieved_k if doc in relevant_docs)
    return relevant_count / k
```

**Recall@K**: What percentage of all relevant docs are in top-K?
```python
def recall_at_k(retrieved_docs, relevant_docs, k):
    retrieved_k = set(retrieved_docs[:k])
    relevant_set = set(relevant_docs)
    return len(retrieved_k & relevant_set) / len(relevant_set)
```

#### 2. Answer Quality

**Faithfulness**: Does the answer use only the provided context?
- Manual review or use LLM-as-judge

**Relevance**: Does the answer address the question?
- Compare to reference answers
- Use semantic similarity

**Completeness**: Does the answer cover all aspects?
- Check against expected key points

#### 3. Performance Metrics

**Latency**: How fast is the system?
```python
import time

start = time.time()
result = rag_system.query(query)
latency = time.time() - start
print(f"Query took {latency:.2f} seconds")
```

**Throughput**: Queries per second

### Test Cases

Create a test suite with various scenarios:

```python
test_cases = [
    {
        "query": "What is machine learning?",
        "expected_keywords": ["algorithm", "data", "learn"],
        "should_cite": True
    },
    {
        "query": "Tell me about quantum computing",
        "expected_response": "no information in context",
        "should_cite": False
    },
    # Edge cases
    {
        "query": "",  # Empty query
        "should_error": True
    },
    {
        "query": "a" * 1000,  # Very long query
        "should_handle": True
    }
]
```

### Automated Testing

```python
def run_tests(rag_system, test_cases):
    results = []
    
    for i, test in enumerate(test_cases):
        print(f"Running test {i+1}/{len(test_cases)}...")
        
        try:
            result = rag_system.query(test['query'])
            
            # Check expected keywords
            if 'expected_keywords' in test:
                answer_lower = result['answer'].lower()
                keywords_found = [
                    kw for kw in test['expected_keywords']
                    if kw in answer_lower
                ]
                passed = len(keywords_found) >= len(test['expected_keywords']) * 0.5
            else:
                passed = True
            
            results.append({
                'test': i+1,
                'passed': passed,
                'answer': result['answer'][:100] + "..."
            })
            
        except Exception as e:
            results.append({
                'test': i+1,
                'passed': test.get('should_error', False),
                'error': str(e)
            })
    
    # Print summary
    passed = sum(1 for r in results if r['passed'])
    print(f"\nTests passed: {passed}/{len(results)}")
    
    return results
```

### Manual Evaluation Rubric

For each test query, rate (1-5):

| Criterion | Score | Notes |
|-----------|-------|-------|
| Relevance | ___/5 | Answer addresses the question |
| Accuracy | ___/5 | Information is correct |
| Completeness | ___/5 | All aspects covered |
| Clarity | ___/5 | Easy to understand |
| Source Usage | ___/5 | Properly cites context |

### Tasks:
1. Create at least 10 test queries (mix of easy, medium, hard)
2. Test edge cases (empty query, out-of-domain query, etc.)
3. Measure average response latency
4. Document test results
5. Identify and fix any issues found

---

In [13]:
# TODO: Create test cases
# Your code here

# test_queries = [
#     "What is AI?",
#     "Explain deep learning",
#     # Add more tests
# ]
# ── Sample document to test against (in case no file uploaded yet) ────────────
sample_text = """
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines.
Machine learning is a subset of AI that enables systems to learn from data without being
explicitly programmed. Deep learning uses neural networks with many layers to analyze data.

Natural Language Processing (NLP) allows computers to understand and generate human language.
Applications include chatbots, translation, and sentiment analysis.

Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.
It retrieves relevant documents from a knowledge base and uses them as context for an LLM
to generate accurate, grounded answers.

Large Language Models (LLMs) are trained on massive text datasets. Examples include
GPT-4, LLaMA, and Mistral. They can perform tasks like summarization, Q&A, and translation.

Vector databases store numerical representations of text called embeddings. FAISS is a
popular open-source vector database developed by Facebook AI Research.
"""

# Load sample text into vector DB if it's empty
if vector_db.index.ntotal == 0:
    print("⚠️ Vector DB is empty — loading sample document for testing...")
    loader  = DocumentLoader()
    chunker = DocumentChunker()
    docs    = loader.load_from_text(sample_text, source_name="sample_ai_doc.txt")
    chunks  = chunker.chunk_documents(docs, strategy="tokens", chunk_size=100, overlap=20)
    vector_db.add_documents(chunks)
    print(f"✅ Sample document loaded — {len(chunks)} chunks indexed.")

# ── 10 Test queries (easy, medium, hard, edge cases) ─────────────────────────
test_cases = [
    # Easy — direct factual
    {
        "id": 1,
        "category": "Easy",
        "query": "What is Artificial Intelligence?",
        "expected_keywords": ["intelligence", "machine", "human"],
        "should_answer": True
    },
    {
        "id": 2,
        "category": "Easy",
        "query": "What is machine learning?",
        "expected_keywords": ["learn", "data", "ai"],
        "should_answer": True
    },
    {
        "id": 3,
        "category": "Easy",
        "query": "What is FAISS?",
        "expected_keywords": ["vector", "facebook", "database"],
        "should_answer": True
    },

    # Medium — multi-concept
    {
        "id": 4,
        "category": "Medium",
        "query": "How does RAG work?",
        "expected_keywords": ["retrieval", "context", "generation"],
        "should_answer": True
    },
    {
        "id": 5,
        "category": "Medium",
        "query": "What are examples of Large Language Models?",
        "expected_keywords": ["gpt", "llama", "mistral"],
        "should_answer": True
    },
    {
        "id": 6,
        "category": "Medium",
        "query": "What are applications of NLP?",
        "expected_keywords": ["chatbot", "translation", "sentiment"],
        "should_answer": True
    },

    # Hard — inferential
    {
        "id": 7,
        "category": "Hard",
        "query": "How are embeddings related to vector databases?",
        "expected_keywords": ["vector", "embedding", "numerical"],
        "should_answer": True
    },
    {
        "id": 8,
        "category": "Hard",
        "query": "What is the relationship between deep learning and AI?",
        "expected_keywords": ["neural", "subset", "layer"],
        "should_answer": True
    },

    # Edge cases
    {
        "id": 9,
        "category": "Edge Case",
        "query": "",  # empty query
        "expected_keywords": [],
        "should_answer": False,
        "is_edge_case": True
    },
    {
        "id": 10,
        "category": "Edge Case",
        "query": "What is the price of gold in 2025?",  # out-of-domain
        "expected_keywords": ["not", "context", "information"],
        "should_answer": True,  # should answer but say "no info"
        "is_edge_case": True
    },
]

print(f"✅ {len(test_cases)} test cases defined!")
print(f"   - Easy: {sum(1 for t in test_cases if t['category']=='Easy')}")
print(f"   - Medium: {sum(1 for t in test_cases if t['category']=='Medium')}")
print(f"   - Hard: {sum(1 for t in test_cases if t['category']=='Hard')}")
print(f"   - Edge Cases: {sum(1 for t in test_cases if t['category']=='Edge Case')}")

⚠️ Vector DB is empty — loading sample document for testing...
⚙️ Generating embeddings for 2 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Added 2 chunks to vector database!
📦 Total vectors in index: 2
✅ Sample document loaded — 2 chunks indexed.
✅ 10 test cases defined!
   - Easy: 3
   - Medium: 3
   - Hard: 2
   - Edge Cases: 2


In [15]:
#TODO: Run tests and measure performance
#chat_completion API with a supported free model
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

def generate_answer(query, context_chunks, max_new_tokens=500, temperature=0.3):
    """Generate answer using chat_completion instead of text_generation"""
    if not context_chunks:
        return "No relevant context found. Please upload a document first."

    # Build context string
    context = "\n\n".join([
        f"[Document {i+1}] (Source: {chunk.get('source','unknown')}, Page: {chunk.get('page','N/A')})\n{chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    # Use messages format (chat_completion)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. Answer ONLY from the provided context. "
                "If the context lacks enough info, say 'I don't have enough information "
                "in the provided context to answer this question.'"
            )
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:"
        }
    ]

    try:
        response = hf_client.chat_completion(
            messages=messages,
            model=LLM_MODEL,
            max_tokens=max_new_tokens,
            temperature=temperature
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        # Fallback: try a different free model
        try:
            response = hf_client.chat_completion(
                messages=messages,
                model="HuggingFaceH4/zephyr-7b-beta",
                max_tokens=max_new_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content.strip()
        except Exception as e2:
            return f"❌ Error generating answer: {str(e2)}"

print("✅ generate_answer() updated to use chat_completion!")
print(f"🤖 Primary model: {LLM_MODEL}")
print("🔄 Fallback model: HuggingFaceH4/zephyr-7b-beta")

✅ generate_answer() updated to use chat_completion!
🤖 Primary model: mistralai/Mistral-7B-Instruct-v0.2
🔄 Fallback model: HuggingFaceH4/zephyr-7b-beta


# ============================================================
# RAG SYSTEM — PROJECT DOCUMENTATION
# ============================================================

## 1. Choice of Embedding Model
- **Model**: `sentence-transformers/all-MiniLM-L6-v2`
- **Dimensions**: 384
- **Reason**: Completely free, runs locally without any API key.
  Lightweight and fast while still producing high-quality semantic
  embeddings. Ideal for a Colab/HF Spaces free-tier environment.

## 2. Choice of LLM
- **Model**: `mistralai/Mistral-7B-Instruct-v0.2`
- **API**: Hugging Face Inference API (free tier)
- **Reason**: Available for free on HF Inference API, supports
  chat_completion format, good instruction-following ability.
  No credit card required — only a free HF account token.
- **Fallback**: `HuggingFaceH4/zephyr-7b-beta` if primary is unavailable.

## 3. Chunking Strategy
- **Method**: Fixed-size token (word-based) chunking
- **Chunk size**: 400 words
- **Overlap**: 50 words (~12.5% overlap)
- **Reason**: Simple, predictable, and effective for general documents.
  Overlap ensures context is not lost at chunk boundaries.
  Sentence-based chunking also implemented as an alternative.

## 4. Vector Database
- **Tool**: FAISS (IndexFlatL2)
- **Reason**: Free, open-source, runs entirely locally. No API key
  or cloud service needed. Fast exact search suitable for small
  to medium document collections.

## 5. Challenges Faced & Solutions

| Challenge | Solution |
|-----------|----------|
| Mistral-7B-Instruct-v0.3 not supported for text_generation | Switched to chat_completion API with v0.2 |
| Free HF API rate limits | Added try/except with fallback model |
| Large PDFs causing memory issues | Chunking with size limit per chunk |
| Empty query edge case | Added input validation before retrieval |
| Out-of-domain queries | Prompt instructs LLM to say "not enough info" |

## 6. RAG Pipeline Summary
1. User uploads PDF or TXT file
2. DocumentLoader reads file page by page
3. DocumentChunker splits into 400-word chunks with 50-word overlap
4. SentenceTransformer encodes chunks into 384-dim vectors
5. FAISS indexes all vectors locally
6. User asks a question in Gradio chat
7. Query is encoded and top-5 similar chunks retrieved via FAISS
8. Chunks + query sent to Mistral-7B via HF Inference API
9. Answer returned with source attribution

## 7. Technologies Used
- gradio — Interactive UI
- faiss-cpu — Vector database
- sentence-transformers — Free local embeddings
- huggingface_hub — Free LLM inference
- pypdf — PDF reading
- numpy — Vector operations
"""

print(documentation)

### Grading Rubric (100 points)

| Component | Points | Criteria |
|-----------|--------|----------|
| **RAG Implementation** | 35 | - Document processing (10)<br>- Vector search (10)<br>- LLM integration (10)<br>- Error handling (5) |
| **Gradio UI** | 25 | - User experience (10)<br>- Feature completeness (10)<br>- Visual design (5) |
| **Deployment** | 20 | - Successful deployment (10)<br>- Works publicly (5)<br>- Good documentation (5) |
| **Testing** | 10 | - Comprehensive tests (5)<br>- Performance metrics (5) |
| **Code Quality** | 10 | - Clean code (5)<br>- Good comments (3)<br>- Proper structure (2) |
| **Bonus** | +10 | Optional improvements implemented |




## Learning Resources

### Official Documentation
- [Gradio Docs](https://www.gradio.app/docs)
- [FAISS Documentation](https://faiss.ai/)
- [Sentence Transformers](https://www.sbert.net/)
- [OpenAI API Reference](https://platform.openai.com/docs)
- [Hugging Face Hub](https://huggingface.co/docs/hub)

### Tutorials
- [LangChain RAG Tutorial](https://python.langchain.com/docs/use_cases/question_answering/)
- [Building RAG from Scratch](https://www.deeplearning.ai/short-courses/building-applications-vector-databases/)
- [Gradio Quickstart](https://www.gradio.app/guides/quickstart)

### Research Papers
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [REALM: Retrieval-Augmented Language Model Pre-Training](https://arxiv.org/abs/2002.08909)

### Community
- [Hugging Face Forums](https://discuss.huggingface.co/)
- [Gradio Discord](https://discord.gg/gradio)
- [r/MachineLearning](https://reddit.com/r/MachineLearning)

---

## Tips for Success

1. **Start Simple**: Get a basic version working first, then add features
2. **Test Often**: Test each component before combining them
3. **Use Free Tiers**: Start with free APIs (Hugging Face) before using paid ones (OpenAI)
4. **Version Control**: Save your work frequently
5. **Ask for Help**: Use the course forum or office hours if stuck
6. **Document As You Go**: Don't wait until the end to write documentation
7. **Time Management**: Allocate 3-4 hours, break it into sessions

---

##  Common Issues & Solutions

### Issue: "FAISS index not working"
**Solution**: Make sure embeddings are `float32` type:
```python
embeddings = embeddings.astype('float32')
index.add(embeddings)
```

### Issue: "Out of memory in Colab"
**Solution**:
- Use smaller embedding models (384d instead of 768d)
- Process documents in batches
- Enable Colab GPU: Runtime → Change runtime type → GPU

### Issue: "Gradio not launching"
**Solution**:
```python
demo.launch(share=True, debug=True)
```

### Issue: "Hugging Face Space build failed"
**Solution**: Check logs for errors, common fixes:
- Update `requirements.txt` versions
- Add `python>=3.8` to requirements
- Check README.md YAML header format

---

**Good luck!**